# Phase 1: Validity checks
**Question:** Is this test valid to read, before any hypothesis is tested?

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.optimize import brentq
from statsmodels.stats.power import NormalIndPower

In [2]:
df = pd.read_csv('cookie_cats.csv')

In [3]:
n_total = len(df)
n_30 = (df['version'] == 'gate_30').sum()
n_40 = (df['version'] == 'gate_40').sum()
print(f"gate_30: {n_30} ({n_30/n_total:.4%})")
print(f"gate_40: {n_40} ({n_40/n_total:.4%})")

chi2_srm, p_srm = stats.chisquare([n_30, n_40], [n_total/2, n_total/2])
print(f"\nChi-square = {chi2_srm:.4f}, p = {p_srm:.4f}")
print("SRM flagged (p < 0.01)" if p_srm < 0.01 else "No SRM flagged")

gate_30: 44700 (49.5626%)
gate_40: 45489 (50.4374%)

Chi-square = 6.9024, p = 0.0086
SRM flagged (p < 0.01)


In [4]:
top5 = df.sort_values('sum_gamerounds', ascending=False).head(5)
print(top5)

        userid  version  sum_gamerounds  retention_1  retention_7
57702  6390605  gate_30           49854        False         True
7912    871500  gate_30            2961         True         True
29417  3271615  gate_40            2640         True        False
43671  4832608  gate_30            2438         True         True
48188  5346171  gate_40            2294         True         True


In [5]:
max_val = df['sum_gamerounds'].max()
second_max = df['sum_gamerounds'].sort_values(ascending=False).iloc[1]
print(f"Ratio to 2nd highest: {max_val / second_max:.1f}x")

Ratio to 2nd highest: 16.8x


In [6]:
q1, q3 = df['sum_gamerounds'].quantile([0.25, 0.75])
fence = q3 + 1.5 * (q3 - q1)
n_above = (df['sum_gamerounds'] > fence).sum()
print(f"IQR upper fence: {fence:.0f} rounds -> {n_above} players above it ({n_above/len(df):.1%})")

IQR upper fence: 120 rounds -> 10177 players above it (11.3%)


In [7]:
df_clean = df[df['sum_gamerounds'] != max_val].copy()
df_clean.to_csv('cookie_cats_clean_for_rounds.csv', index=False)

In [8]:
print(f"Raw: {len(df)} rows -> Clean: {len(df_clean)} rows")

Raw: 90189 rows -> Clean: 90188 rows


In [9]:
alpha, target_power = 0.05, 0.80
n_per_arm = min(n_30, n_40)                      # conservative: smaller arm
baseline = df.loc[df['version'] == 'gate_30', 'retention_7'].astype(int).mean()

pa = NormalIndPower()
h_mde = pa.solve_power(effect_size=None, nobs1=n_per_arm, alpha=alpha, power=target_power,
                       ratio=1.0, alternative='two-sided')
p2 = brentq(lambda p: (2*np.arcsin(np.sqrt(baseline)) - 2*np.arcsin(np.sqrt(p))) - h_mde, 0.001, baseline)
print(f"Players per arm: {n_per_arm} | baseline 7-day retention: {baseline:.2%}")
print(f"Minimum detectable effect: {(baseline - p2)*100:.2f} percentage points (Cohen's h = {h_mde:.4f})")

g40_rate = df.loc[df['version'] == 'gate_40', 'retention_7'].astype(int).mean()
h_obs = 2*np.arcsin(np.sqrt(baseline)) - 2*np.arcsin(np.sqrt(g40_rate))
power_obs = pa.solve_power(effect_size=h_obs, nobs1=n_per_arm, alpha=alpha, power=None,
                           ratio=1.0, alternative='two-sided')
print(f"Observed gap: {(baseline - g40_rate)*100:.2f} pp -> achieved power: {power_obs:.1%}")

Players per arm: 44700 | baseline 7-day retention: 19.02%
Minimum detectable effect: 0.73 percentage points (Cohen's h = 0.0187)
Observed gap: 0.82 pp -> achieved power: 88.3%
